# nnU-Net BraTS 2024 MEN-RT — Full Pipeline

**Before running this notebook:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset (the two zip files) as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `GITHUB_TOKEN`
   - Notebook → Add-ons → Secrets → Add New Secret

**Training plan (pilot — 50 cases subset):**
- Fold 0: 50 epochs → then run inference (fold 0 only, not ensemble) → prompts for foundation model
- Fold 1: 50 epochs (Cell 17, same session or new session)
- Early stopping is active but only kicks in after the 50-epoch warmup

**Checkpoints saved automatically per fold:**
- `checkpoint_best.pth` — best validation Dice
- `checkpoint_latest.pth` — most recent epoch (resume after interruption)
- `checkpoint_final.pth` — written at end of training

**Expected runtime:** ~1–2 hours on T4 GPU for 50 epochs / 50 cases

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Check available input data ───────────────────────────────────────
# Run this to find the correct path to your BraTS zip files
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets  = UserSecretsClient()
token    = secrets.get_secret('nnunet_kaggle')
repo_url = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'

# Remove old clone if exists, then fresh clone
os.system('rm -rf /kaggle/working/nnunet-training')
os.system(f'git clone {repo_url} /kaggle/working/nnunet-training')

print('
Repository contents:')
os.system('ls /kaggle/working/nnunet-training')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
# surface-distance must be installed from GitHub (PyPI version is a different package)
!pip install git+https://github.com/google-deepmind/surface-distance.git -q
!pip install nnunetv2 -q
!pip install nibabel SimpleITK scipy scikit-image pandas matplotlib loguru python-dotenv tqdm rich mlflow -q

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Configure environment variables ───────────────────────────────────
#
# !! EDIT THE TWO PATHS BELOW !!
# Check Cell 2 output to find the correct paths to your zip files.
# Example: /kaggle/input/brats-men-rt/BraTS2024-MEN-RT-TrainingData.zip
#
TRAIN_ZIP = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-TrainingData.zip'
VAL_ZIP   = '/kaggle/input/YOUR-DATASET-NAME/BraTS2024-MEN-RT-ValidationData.zip'

# ── Subset size (set to None to use ALL cases) ────────────────────────────────
MAX_CASES = 50    # train on only the first 50 cases for this pilot run

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT = '/kaggle/working/nnunet-training'

os.environ['nnUNet_raw']          = f'{PROJECT}/nnunet_raw'
os.environ['nnUNet_preprocessed'] = f'{PROJECT}/nnunet_preprocessed'
os.environ['nnUNet_results']      = f'{PROJECT}/checkpoints'
os.environ['DATASET_ID']          = '001'
os.environ['DATASET_NAME']        = 'BraTSMENRT'
os.environ['NNUNET_CONFIGURATION']= '3d_fullres'
os.environ['NNUNET_SEED']         = '42'
os.environ['ES_PATIENCE']         = '50'
os.environ['ES_MIN_DELTA']        = '0.0001'
os.environ['ES_WARMUP']           = '50'
os.environ['NUM_FOLDS']           = '5'
os.environ['CUDA_VISIBLE_DEVICES']= '0'
os.environ['TRAIN_SOURCE']        = TRAIN_ZIP
os.environ['VAL_SOURCE']          = VAL_ZIP
os.environ['LABEL_SUFFIX']        = 'gtv'
os.environ['EXPERIMENT_NAME']     = 'BraTSMENRT_baseline'
os.environ['MLFLOW_TRACKING_URI'] = f'{PROJECT}/experiments/mlruns'

# Create output directories
for d in ['nnunet_raw','nnunet_preprocessed','checkpoints',
          'metrics','results','visualizations','inference_outputs','logs','experiments']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

print('Environment configured:')
print(f'  nnUNet_raw          : {os.environ["nnUNet_raw"]}')
print(f'  nnUNet_preprocessed : {os.environ["nnUNet_preprocessed"]}')
print(f'  nnUNet_results      : {os.environ["nnUNet_results"]}')
print(f'  TRAIN_SOURCE        : {TRAIN_ZIP}')
print(f'  VAL_SOURCE          : {VAL_ZIP}')
print(f'  MAX_CASES           : {MAX_CASES}')

In [ ]:
# ── Cell 6: Change working directory to the repo ──────────────────────────────
# All ! commands after this run inside the repository folder
%cd /kaggle/working/nnunet-training

In [ ]:
# ── Cell 7: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────────
# Uses only the first MAX_CASES training cases (pilot run = 50 cases).
# Set MAX_CASES = None in Cell 5 to use the full dataset.
# Expected time: 2–5 minutes for 50 cases
#
# NOTE: Explicit PYTHONPATH avoids ModuleNotFoundError: No module named 'src.data'
!PYTHONPATH=/kaggle/working/nnunet-training:$PYTHONPATH python scripts/01_prepare_dataset.py \
    --train    "{TRAIN_ZIP}" \
    --val      "{VAL_ZIP}"   \
    --max-cases {MAX_CASES}  \
    --log-dir  logs

In [ ]:
# Verify dataset was created correctly
!ls -lh $nnUNet_raw/Dataset001_BraTSMENRT/
!echo '--- imagesTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/imagesTr/ | wc -l
!echo '--- labelsTr count ---'
!ls $nnUNet_raw/Dataset001_BraTSMENRT/labelsTr/ | wc -l
!echo '--- dataset.json ---'
!cat $nnUNet_raw/Dataset001_BraTSMENRT/dataset.json

In [ ]:
# ── Cell 8: STEP 2 — nnU-Net Planning and Preprocessing ──────────────────────
# nnU-Net reads the data and automatically decides:
#   - patch size, batch size, network depth
#   - resampling spacing, normalization
# Expected time: 20–40 minutes
!python scripts/02_preprocess.py --log-dir logs

In [ ]:
# Verify preprocessing output
!ls $nnUNet_preprocessed/Dataset001_BraTSMENRT/
!echo '--- Splits file (5-fold CV) ---'
!python -c "import json; s=json.load(open('$nnUNet_preprocessed/Dataset001_BraTSMENRT/splits_final.json')); print(f'{len(s)} folds | fold_0 train={len(s[0][\"train\"])} val={len(s[0][\"val\"])}')" 2>/dev/null || echo 'splits_final.json not found yet'

In [ ]:
# ── Cell 8b (NEW): Integrity Check — Verify 50-case Subset ─────────────────────
# 
# 🔍 CRITICAL VERIFICATION STEP
# This cell ensures your 50-case subset is valid before training.
# It checks:
#   1. dataset.json has correct numTraining (should be 50)
#   2. All image files are readable NIfTI format
#   3. All labels are present and valid (binary: {0, 1})
#   4. Image and label shapes match
#   5. No duplicate case IDs
#
# If this cell fails, DO NOT proceed to training. Fix the issues first.

import sys
sys.path.insert(0, '/kaggle/working/nnunet-training')

from pathlib import Path
from src.data.integrity_checker import IntegrityChecker

dataset_dir = Path(os.environ['nnUNet_raw']) / 'Dataset001_BraTSMENRT'
print(f'Checking dataset: {dataset_dir}\n')

checker = IntegrityChecker(dataset_dir=dataset_dir, expected_label_values={0, 1})
report = checker.run(max_cases=None)  # Check ALL 50 cases

# Print summary
print('\n' + '='*70)
print('  INTEGRITY CHECK REPORT')
print('='*70)
print(report.summary())
print('='*70 + '\n')

# Print any errors
if report.case_reports:
    failed = [r for r in report.case_reports if not r.ok]
    if failed:
        print(f'⚠️  {len(failed)} cases have issues:\n')
        for r in failed:
            print(f'  {r.case_id} ({r.split}):')
            for err in r.errors:
                print(f'    • {err}')
        print('\n❌ ERROR: Fix the above issues before proceeding to training!\n')
        assert not failed, "Dataset integrity check failed"
    else:
        print('✅ All cases passed integrity checks!')
else:
    print('⚠️  No cases found to check')

# Verify numTraining matches actual count
import json
dataset_json_path = dataset_dir / 'dataset.json'
with open(dataset_json_path) as f:
    ds_json = json.load(f)

num_training = ds_json.get('numTraining', -1)
print(f'\nDataset metadata:')
print(f'  numTraining in dataset.json  : {num_training}')
print(f'  Actual training cases found  : {report.num_train_cases}')
print(f'  Match? {num_training == report.num_train_cases} ✓' if num_training == report.num_train_cases else f'  Match? NO ✗ (MISMATCH!)')
print(f'  Channel names                : {ds_json.get("channel_names")}')
print(f'  Labels                       : {ds_json.get("labels")}')

if num_training != report.num_train_cases:
    raise ValueError(
        f'numTraining mismatch! dataset.json says {num_training} but found {report.num_train_cases} cases. '
        'This MUST be fixed before preprocessing.'
    )

print('\n✅ Integrity check PASSED. Safe to proceed to training.')


In [ ]:
# ── Cell 9: STEP 3 — Train Fold 0 (50 epochs) ───────────────────────────────
#
# Trains Fold 0 on the 50-case subset for 50 epochs.
# nnU-Net automatically saves:
#   checkpoint_best.pth   ← best validation Dice (saved whenever Dice improves)
#   checkpoint_latest.pth ← saved every epoch  (safe to resume if interrupted)
#   checkpoint_final.pth  ← written at the end of training
#
# Early stopping: patience=50, warmup=50, min_delta=0.0001
# (with only 50 epochs total it won't trigger, but it's wired in)
#
# If interrupted, re-run Cells 1-6, then add --c to resume from checkpoint_latest:
#   !nnUNetv2_train 001 3d_fullres 0 -tr nnUNetTrainerEarlyStopping -p nnUNetPlans --num_epochs 50 --c
#
# Expected time: ~1-2 hours on T4 GPU
#
!nnUNetv2_train 001 3d_fullres 0 \n
    -tr nnUNetTrainerEarlyStopping \n
    -p  nnUNetPlans \n
    --num_epochs 50


---

## Training Phase: Fold 0 (50 epochs)

**Before you start:**

### About Early Stopping & Training Duration

The training uses **early stopping with a 50-epoch warmup period**:
- **ES_PATIENCE = 50** — stops if no improvement for 50 epochs
- **ES_WARMUP = 50** — early stopping is inactive for first 50 epochs
- **NUM_EPOCHS = 50** — we're training exactly 50 epochs

⚠️ **Important**: Since we're training **exactly 50 epochs and early stopping only activates after 50 epochs**, early stopping will **NOT trigger on this pilot run**. It's configured for future full training.

### Expected Timeline

- **Preprocessing time**: 20-40 minutes (already done)
- **Fold 0 training**: 1-2 hours on T4 GPU (or 30-40 min on P100)
- **Inference**: 10-30 minutes
- **Evaluation**: 5-15 minutes
- **Total**: ~2-4 hours

### Checkpoints Saved

During training, nnU-Net automatically saves:
- `checkpoint_best.pth` — Updated whenever validation Dice improves
- `checkpoint_latest.pth` — Updated every epoch (for resuming)
- `checkpoint_final.pth` — Saved at end of training

---

In [ ]:
# Verify checkpoints were saved
import os
CKPT_DIR = '/kaggle/working/nnunet-training/checkpoints/Dataset001_BraTSMENRT/nnUNetTrainerEarlyStopping__nnUNetPlans__3d_fullres/fold_0'
print(f'Checkpoint directory: {CKPT_DIR}')
!ls -lh "{CKPT_DIR}" 2>/dev/null || echo 'Not found — training may have failed'
!echo ''
!echo 'Expected files:'
!echo '  checkpoint_best.pth   <- best val Dice'
!echo '  checkpoint_latest.pth <- most recent epoch (use --c to resume)'
!echo '  checkpoint_final.pth  <- written at end of training'

# Print latest training log metrics
import glob
log_files = sorted(glob.glob(f'{CKPT_DIR}/training_log*.txt'))
if log_files:
    print(f'
Last 20 lines of {log_files[-1]}:')
    with open(log_files[-1]) as fh:
        lines = fh.readlines()
    print(''.join(lines[-20:]))


In [ ]:
# ── Cell 9b (NEW): Checkpoint Validation — Verify Saves ──────────────────────
# 
# ✅ Validates that training saved checkpoints correctly.
# This must pass before running inference!
# 
# Checks:
#   1. checkpoint_best.pth exists and has size > 0
#   2. checkpoint_final.pth exists and has size > 0
#   3. metadata.json was written with training metrics
#
# If this fails: training may have been interrupted or crashed.
#                Re-run Cell 9 to complete or resume with: --c flag

import sys
sys.path.insert(0, '/kaggle/working/nnunet-training')

from pathlib import Path
from src.training.checkpoint_manager import CheckpointManager

ckpt_mgr = CheckpointManager(root='checkpoints')
available = ckpt_mgr.list_available()

print('='*70)
print('  CHECKPOINT VALIDATION')
print('='*70)
print()

if not available:
    print('❌ ERROR: No checkpoints found!')
    print('   Training may not have completed. Check the logs above.')
    raise RuntimeError('Checkpoint validation failed: no checkpoints found')

fold_key = 'fold_0'
if fold_key not in available:
    print(f'❌ ERROR: {fold_key} not found')
    print(f'   Available: {list(available.keys())}')
    raise RuntimeError(f'Checkpoint validation failed: {fold_key} missing')

files = available[fold_key]
print(f'Found fold_0 checkpoints: {files}')
print()

# Check mandatory files
mandatory = ['best_model.pth', 'last_model.pth']
missing = [f for f in mandatory if f not in files]

if missing:
    print(f'❌ MISSING checkpoints: {missing}')
    print('   Training may have failed. Check logs above.')
    raise RuntimeError(f'Checkpoint validation failed: missing {missing}')

# Check file sizes
for fname in mandatory:
    path = Path('checkpoints') / fold_key / fname
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f'  ✓ {fname:25} {size_mb:8.1f} MB')

# Check metadata
meta_path = Path('checkpoints') / fold_key / 'metadata.json'
if meta_path.exists():
    import json
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'  ✓ metadata.json')
    if 'best_val_dice' in meta:
        print(f'    → best_val_dice: {meta["best_val_dice"]:.4f}')
    if 'epochs_trained' in meta:
        print(f'    → epochs_trained: {meta["epochs_trained"]}')
else:
    print(f'  ⚠️ metadata.json not found (optional)')

print()
print('='*70)
print('  ✅ CHECKPOINT VALIDATION PASSED')
print('='*70)
print()
print('Safe to proceed to Step 4: Inference')


In [ ]:
# ── Cell 10: STEP 4 — Inference using Fold 0 only ────────────────────────────
#
# Predicts GTV masks for the held-out validation cases of fold 0,
# using ONLY fold 0's trained model (no ensemble, no other folds).
#
# These predictions will be used as segmentation prompts for the next
# foundation model to refine segmentation.
#
# Output: inference_outputs/cv/fold_0/*.nii.gz
# Expected time: 10–30 minutes
!PYTHONPATH=/kaggle/working/nnunet-training:$PYTHONPATH python scripts/04_inference.py \
    --cv-mode \
    --folds 0 \
    --output inference_outputs/cv \
    --log-dir logs

In [ ]:
# Verify fold 0 prompt predictions exist
!echo '--- Fold 0 prompt predictions (for foundation model) ---'
!ls inference_outputs/cv/fold_0/ 2>/dev/null | head -10
!echo ''
!echo 'Total predictions (fold 0 val cases):'
!ls inference_outputs/cv/fold_0/*.nii.gz 2>/dev/null | wc -l
!echo ''
!echo '--- Manifest ---'
!cat inference_outputs/cv/cv_prediction_manifest.json 2>/dev/null || echo 'Manifest not found'

In [ ]:
# ── Cell 11: STEP 5 — Evaluate Fold 0 ────────────────────────────────────────
# Computes: DSC, HD95, NSD, Precision, Recall, Specificity, Volume Error
# Expected time: 5–15 minutes
!PYTHONPATH=/kaggle/working/nnunet-training:$PYTHONPATH python scripts/05_evaluate.py \
    --cv-mode \
    --results-dir results \
    --show-rankings \
    --log-dir logs

In [ ]:
# ── Cell 12: STEP 6 — Visualizations ─────────────────────────────────────────
# Generates: overlays, violin plots, fold comparison charts, training curves
# Expected time: 2–5 minutes
!PYTHONPATH=/kaggle/working/nnunet-training:$PYTHONPATH python scripts/06_visualize.py \
    --all \
    --cv-mode \
    --results-dir results \
    --metrics-dir metrics \
    --output-dir visualizations \
    --log-dir logs

In [ ]:
# ── Cell 13: View Results ─────────────────────────────────────────────────────
import pandas as pd

result_files = [
    'results/fold_0_per_case.csv',
    'results/cv_combined.csv',
]

for f in result_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        cols = ['case_id','dice','hd95','nsd','precision','recall']
        cols = [c for c in cols if c in df.columns]
        print(f'\n=== {f} ({len(df)} cases) ===')
        print(df[cols].round(4).to_string(index=False))
        print(f'\nMean DSC  : {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
        if 'hd95' in df.columns:
            finite_hd = df['hd95'].replace([float('inf')], float('nan'))
            print(f'Mean HD95 : {finite_hd.mean():.2f} mm')
        if 'nsd' in df.columns:
            print(f'Mean NSD  : {df["nsd"].mean():.4f}')
    else:
        print(f'{f} not found yet')

In [ ]:
# ── Cell 14: Show overlay images ──────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

overlay_dir = Path('visualizations/overlays')
if overlay_dir.exists():
    images = sorted(overlay_dir.glob('*.png'))[:3]  # show first 3
    for img_path in images:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))
else:
    print('No overlay images found — run Cell 12 first')

In [ ]:
# ── Cell 15: Show metric plots ────────────────────────────────────────────────
for plot in ['visualizations/metrics_violin.png',
             'visualizations/metrics_boxplot.png',
             'visualizations/volume_scatter.png']:
    if os.path.exists(plot):
        print(plot)
        display(Image(plot, width=900))

In [ ]:
# ── Cell 16: Save everything to Kaggle output ─────────────────────────────────
# Kaggle deletes /kaggle/working/nnunet-training when the session ends.
# This cell copies all important outputs to /kaggle/working/ which IS saved.
import shutil, os

SAVE = '/kaggle/working/outputs'
os.makedirs(SAVE, exist_ok=True)

to_save = [
    ('results',                        f'{SAVE}/results'),
    ('metrics',                        f'{SAVE}/metrics'),
    ('visualizations',                 f'{SAVE}/visualizations'),
    ('inference_outputs',              f'{SAVE}/inference_outputs'),
    # Fold 0 only predictions — prompts for foundation model
    ('inference_outputs/fold0_prompts',f'{SAVE}/fold0_prompts'),
    # Checkpoints: checkpoint_best.pth, checkpoint_latest.pth, checkpoint_final.pth
    ('checkpoints',                    f'{SAVE}/checkpoints'),
]

for src, dst in to_save:
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {src} -> {dst}')
    else:
        print(f'Skipped (not found): {src}')

print('
All outputs saved to /kaggle/working/outputs/')
!du -sh /kaggle/working/outputs/*

# Show checkpoint files per fold
import glob
ckpt_root = f'{SAVE}/checkpoints'
for ckpt in sorted(glob.glob(f'{ckpt_root}/**/*.pth', recursive=True)):
    print(f'  Checkpoint: {ckpt}')

---

## After Fold 0: Inference → Foundation Model Prompts

After Cell 9 (train fold 0) completes:
1. Run **Cell 10** → inference with fold 0 model only → `inference_outputs/fold0_prompts/fold_0/*.nii.gz`
2. These `.nii.gz` files are your **segmentation prompts** for the next foundation model
3. Run Cells 11–16 to evaluate metrics and save outputs

## Train Fold 1 (Cell 17 below)

Run Cell 17 **after fold 0 + inference are done**.
If in a **new Kaggle session**: re-run Cells 1–6 (skip 7–8 if already preprocessed), then run Cell 17.

> **Checkpoint resume tip:** If training is interrupted, add `--c` flag:
> ```
> nnUNetv2_train 001 3d_fullres 0 -tr nnUNetTrainerEarlyStopping --num_epochs 50 --c
> ```
> nnU-Net will resume from `checkpoint_latest.pth` automatically.

In [ ]:
# ── Cell 17: Train Fold 1 (50 epochs) ────────────────────────────────────────
#
# ⚠️ IMPORTANT: This cell uses the orchestrator script (scripts/03_train.py)
# for consistency with logging, checkpointing, and metric tracking.
#
# Run this cell AFTER fold 0 training + inference are done.
# Trains fold 1 for 50 epochs with the same settings as fold 0.
# Checkpoints are saved to: checkpoints/fold_1/{best,last}_model.pth
# Metrics logged to: metrics/fold_1_training_log.csv
#
# If running in a new session: re-run Cells 1–8 first, then run this cell.
#
# Resuming interrupted training:
#   python scripts/03_train.py --folds 1 --continue-training
#

import subprocess
import sys
import os

# Ensure we're in the repo directory
os.chdir('/kaggle/working/nnunet-training')

FOLD = 1
print(f'\n{"="*70}')
print(f'  FOLD {FOLD}: Training 50 epochs')
print(f'{"="*70}\n')

# Use the orchestrator for consistent logging and checkpointing
cmd = [
    sys.executable,
    'scripts/03_train.py',
    '--folds', str(FOLD),
    '--seed', '42',
    '--es-patience', '50',
    '--es-min-delta', '0.0001',
    '--es-warmup', '50',
    '--log-dir', 'logs',
    '--metrics-dir', 'metrics',
    '--checkpoints-dir', 'checkpoints',
]

print(f'Running: {" ".join(cmd)}\n')
result = subprocess.run(cmd)

if result.returncode != 0:
    print(f'\n❌ Fold {FOLD} training FAILED with exit code {result.returncode}')
    sys.exit(1)
else:
    print(f'\n✅ Fold {FOLD} training COMPLETED successfully')
    print(f'   Checkpoints: checkpoints/fold_{FOLD}/')
    print(f'   Metrics: metrics/fold_{FOLD}_training_log.csv')
    print(f'\nNext steps:')
    print(f'  1. Run Cell 18: Inference for fold {FOLD}')
    print(f'  2. Or proceed to Cell 19: Train fold {FOLD+1}')